# 04 - Feature Engineering en Spark (PySpark)

Pipeline escalable de ingesta y construcción de features sobre el dataset OULAD.
Replica la lógica de `02_features.ipynb` en PySpark sobre Databricks Community Edition.

**Ruta de datos:** /Workspace/Users/bsolanahurtado@gmail.com/TFM-OULAD/
**Output:** df_modelo_spark.csv (27.553 filas × 25 columnas)
**Justificación Spark:** arquitectura preparada para producción, no por volumen del CSV.

## 0. Setup y carga de tablas OULAD

In [0]:
from pyspark.sql import functions as F

PATH = "/Workspace/Users/bsolanahurtado@gmail.com/TFM-OULAD/"

courses         = spark.read.csv(PATH + "courses.csv",             header=True, inferSchema=True)
assessments     = spark.read.csv(PATH + "assessments.csv",         header=True, inferSchema=True)
vle             = spark.read.csv(PATH + "vle.csv",                 header=True, inferSchema=True)
student_info    = spark.read.csv(PATH + "studentInfo.csv",         header=True, inferSchema=True)
student_reg     = spark.read.csv(PATH + "studentRegistration.csv", header=True, inferSchema=True)
student_assess  = spark.read.csv(PATH + "studentAssessment.csv",   header=True, inferSchema=True)
student_vle     = spark.read.csv(PATH + "studentVle.csv",          header=True, inferSchema=True)

# Verificación
tablas = {
    "courses": (courses, 22),
    "assessments": (assessments, 206),
    "vle": (vle, 6364),
    "student_info": (student_info, 32593),
    "student_reg": (student_reg, 32593),
    "student_assess": (student_assess, 173912),
    "student_vle": (student_vle, 10655280),
}

for nombre, (df, esperado) in tablas.items():
    n = df.count()
    estado = "✅" if n == esperado else f"⚠️  esperado {esperado}"
    print(f"{estado}  {nombre}: {n:,} filas | {len(df.columns)} cols")

✅  courses: 22 filas | 3 cols
✅  assessments: 206 filas | 6 cols
✅  vle: 6,364 filas | 6 cols
✅  student_info: 32,593 filas | 12 cols
✅  student_reg: 32,593 filas | 5 cols
✅  student_assess: 173,912 filas | 5 cols
✅  student_vle: 10,655,280 filas | 6 cols


## 1. Construir df_activos

In [0]:
# Join student_info + student_reg
df = student_info.join(
    student_reg.select("code_module", "code_presentation", "id_student", "date_unregistration"),
    on=["code_module", "code_presentation", "id_student"],
    how="left"
)

# Filtro de activos: excluir date_unregistration <= 27
df_activos = df.filter(
    F.col("date_unregistration").isNull() | (F.col("date_unregistration") > 27)
)

# Variable riesgo
df_activos = df_activos.withColumn(
    "riesgo",
    F.when(F.col("final_result").isin("Fail", "Withdrawn"), 1).otherwise(0)
)

# Corrección imd_band (misma que en pandas)
df_activos = df_activos.withColumn(
    "imd_band",
    F.when(F.col("imd_band") == "10-20", "10-20%")
     .when(F.col("imd_band").isNull(), "Missing")
     .otherwise(F.col("imd_band"))
)

# Verificación
n = df_activos.count()
n_riesgo = df_activos.filter(F.col("riesgo") == 1).count()
print(f"df_activos: {n:,} estudiantes")
print(f"Riesgo=1:   {n_riesgo:,} ({n_riesgo/n*100:.1f}%)")
print(f"Riesgo=0:   {n - n_riesgo:,} ({(n - n_riesgo)/n*100:.1f}%)")

df_activos: 27,553 estudiantes
Riesgo=1:   12,168 (44.2%)
Riesgo=0:   15,385 (55.8%)


## 2. Dedup studentVle + features VLE

In [0]:
# 1. Eliminar duplicados exactos (las 6 columnas idénticas)
student_vle_clean = student_vle.dropDuplicates()

# Verificación intermedia — debe dar ~9.868.110 (10.655.280 - 787.170)
print(f"Tras dropDuplicates: {student_vle_clean.count():,} filas")

# 2. Filtrar ventana 0-27 y pre-curso por separado
vle_ventana  = student_vle_clean.filter((F.col("date") >= 0) & (F.col("date") <= 27))
vle_precurso = student_vle_clean.filter(F.col("date") < 0)

# 3. Agregar clics por semana en ventana (groupBy suma los sum_click legítimos)
vle_agg = vle_ventana.groupBy("id_student", "code_module", "code_presentation").agg(
    F.sum("sum_click").alias("total_clics"),
    F.countDistinct("date").alias("dias_activo"),
    F.sum(F.when((F.col("date") >= 0)  & (F.col("date") <= 6),  F.col("sum_click"))).alias("clics_semana_1"),
    F.sum(F.when((F.col("date") >= 7)  & (F.col("date") <= 13), F.col("sum_click"))).alias("clics_semana_2"),
    F.sum(F.when((F.col("date") >= 14) & (F.col("date") <= 20), F.col("sum_click"))).alias("clics_semana_3"),
    F.sum(F.when((F.col("date") >= 21) & (F.col("date") <= 27), F.col("sum_click"))).alias("clics_semana_4"),
)

# 4. Clics pre-curso
precurso_agg = vle_precurso.groupBy("id_student", "code_module", "code_presentation").agg(
    F.sum("sum_click").alias("clics_precurso")
)

# 5. Regularidad y tipos_actividad (requiere join con vle para activity_type)
tipos_agg = vle_ventana.join(
    vle.select("id_site", "activity_type"),
    on="id_site", how="left"
).groupBy("id_student", "code_module", "code_presentation").agg(
    F.countDistinct("activity_type").alias("tipos_actividad")
)

# 6. Unir todo con df_activos (left join para conservar estudiantes sin actividad → quedan como null → luego a 0)
df_vle = df_activos.join(vle_agg,    on=["id_student", "code_module", "code_presentation"], how="left") \
                   .join(precurso_agg, on=["id_student", "code_module", "code_presentation"], how="left") \
                   .join(tipos_agg,    on=["id_student", "code_module", "code_presentation"], how="left")

# 7. Rellenar nulls con 0 (estudiantes sin ninguna actividad en ventana)
cols_vle = ["total_clics", "dias_activo", "clics_semana_1", "clics_semana_2",
            "clics_semana_3", "clics_semana_4", "clics_precurso", "tipos_actividad"]
df_vle = df_vle.fillna(0, subset=cols_vle)

# 8. Añadir regularidad
df_vle = df_vle.withColumn("regularidad", F.col("dias_activo") / 28.0)

# Verificación final
print(f"df_vle: {df_vle.count():,} filas (debe ser 27.553)")
df_vle.select(cols_vle).describe().show()

Tras dropDuplicates: 9,868,110 filas
df_vle: 27,553 filas (debe ser 27.553)
+-------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+
|summary|       total_clics|       dias_activo|    clics_semana_1|    clics_semana_2|    clics_semana_3|   clics_semana_4|    clics_precurso|   tipos_actividad|
+-------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+
|  count|             27553|             27553|             27553|             27553|             27553|            27553|             27553|             27553|
|   mean|267.04533081697093|10.653068631365006| 68.28131237977716|61.379994918883604| 79.44485174028236| 57.9391717780278| 72.78347185424455| 6.483468224875694|
| stddev| 322.6311237084891| 7.192714636270607|108.00757405646236| 92.12675099887119|105.20436986454892|92.278078825541

## 3. Features de evaluación

In [0]:
# 1. Join student_assess + assessments para obtener code_module y code_presentation
assess_join = student_assess.join(
    assessments.select("id_assessment", "code_module", "code_presentation"),
    on="id_assessment",
    how="left"
)

# 2. Filtrar: solo entregas dentro de ventana, no bancadas
assess_ventana = assess_join.filter(
    (F.col("date_submitted") <= 27) & (F.col("is_banked") == 0)
)

# 3. Agregar por estudiante-módulo-presentación
assess_agg = assess_ventana.groupBy("id_student", "code_module", "code_presentation").agg(
    F.count("id_assessment").alias("n_entregas_ventana"),
    F.avg("score").alias("nota_media_ventana"),
    F.min("score").alias("nota_min_ventana")
)

# 4. Left join sobre df_vle
df_features = df_vle.join(
    assess_agg,
    on=["id_student", "code_module", "code_presentation"],
    how="left"
)

# 5. entrego_algo + rellenar nulls de conteo (nota se deja null para CatBoost)
df_features = df_features.withColumn(
    "entrego_algo",
    F.when(F.col("n_entregas_ventana") >= 1, 1).otherwise(0)
).fillna(0, subset=["n_entregas_ventana", "entrego_algo"])

# Verificación
print(f"df_features: {df_features.count():,} filas (debe ser 27.553)")
df_features.select("n_entregas_ventana", "entrego_algo", "nota_media_ventana", "nota_min_ventana") \
           .describe().show()

print("\nEntregas por módulo:")
df_features.groupBy("code_module").agg(
    F.avg("n_entregas_ventana").alias("media_entregas"),
    F.count(F.when(F.col("nota_media_ventana").isNull(), 1)).alias("nulls_nota")
).orderBy("code_module").show()

df_features: 27,553 filas (debe ser 27.553)
+-------+------------------+------------------+------------------+------------------+
|summary|n_entregas_ventana|      entrego_algo|nota_media_ventana|  nota_min_ventana|
+-------+------------------+------------------+------------------+------------------+
|  count|             27553|             27553|             19782|             19782|
|   mean|0.8353355351504373|0.7181795085834574| 73.04990648063894| 72.22677181275907|
| stddev|0.6393366520368509|0.4498944854741498| 21.73720726091884|21.782097455457944|
|    min|                 0|                 0|               0.0|                 0|
|    max|                 4|                 1|             100.0|               100|
+-------+------------------+------------------+------------------+------------------+


Entregas por módulo:
+-----------+--------------------+----------+
|code_module|      media_entregas|nulls_nota|
+-----------+--------------------+----------+
|        AAA|  0.9025

## 4. Ensamblado final y exportación

In [0]:
# Columnas finales — exactamente las mismas que df_modelo.csv (25 cols)
columnas_modelo = [
    # Identificadores / target
    "id_student", "code_module", "code_presentation", "riesgo",
    # Demográficas
    "gender", "region", "highest_education", "imd_band",
    "age_band", "num_of_prev_attempts", "studied_credits", "disability",
    # VLE
    "total_clics", "dias_activo", "clics_semana_1", "clics_semana_2",
    "clics_semana_3", "clics_semana_4", "clics_precurso",
    "regularidad", "tipos_actividad",
    # Assessment
    "n_entregas_ventana", "entrego_algo", "nota_media_ventana", "nota_min_ventana"
]

df_modelo_spark = df_features.select(columnas_modelo)

# Verificación anti-leakage: estas columnas NO deben estar
cols_prohibidas = ["final_result", "date_unregistration", "date_registration"]
cols_presentes = [c for c in cols_prohibidas if c in df_modelo_spark.columns]
if cols_presentes:
    print(f"⚠️  LEAKAGE DETECTADO: {cols_presentes}")
else:
    print("✅ Sin leakage — columnas prohibidas ausentes")

# Shape final
print(f"Shape: {df_modelo_spark.count():,} filas × {len(df_modelo_spark.columns)} columnas")

# Exportar a CSV en el Workspace
OUTPUT = "/Workspace/Users/bsolanahurtado@gmail.com/TFM-OULAD/df_modelo_spark.csv"
df_modelo_spark.coalesce(1).write.csv(OUTPUT, header=True, mode="overwrite")
print(f"✅ Exportado en {OUTPUT}")

✅ Sin leakage — columnas prohibidas ausentes
Shape: 27,553 filas × 25 columnas
✅ Exportado en /Workspace/Users/bsolanahurtado@gmail.com/TFM-OULAD/df_modelo_spark.csv
